# Phase 3 — Churn Intelligence (XGBoost + Cox + Uplift)

Builds a full churn intelligence stack on top of the Phase 2 CLV scores.

**Sections**
1. Setup & data preparation
2. Step 3.1 — Define churn label
3. Step 3.2 — XGBoost churn classifier
4. Step 3.3 — SHAP explanations
5. Step 3.4 — Cox Proportional Hazards survival model
6. Step 3.5 — Uplift model (T-Learner)
7. Step 3.6 — Merge full intelligence table

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from lifelines import CoxPHFitter, KaplanMeierFitter

from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, build_master_customer_table,
)
from models.churn_model import (
    define_churn_label, prepare_churn_features,
    fit_xgboost_classifier, evaluate_classifier,
    get_shap_values, get_top_shap_drivers,
    fit_cox_model, predict_survival_days,
    compute_treatment_proxy, fit_uplift_tlearner, score_uplift,
    build_full_intelligence_table,
    CHURN_FEATURES,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, df_all = clean_data(df_raw)
master = build_master_customer_table(df_customers, return_features, cancel_features)

clv_scores = pd.read_csv('../data/processed/clv_scores.csv')
print(f'\nmaster shape     : {master.shape}')
print(f'clv_scores shape : {clv_scores.shape}')

## Step 3.1 — Define Churn Label

**Definition:** `churned = 1` if `recency > 180 days`, else `0`.

Recency is the number of days since the customer's last purchase, measured from the dataset snapshot date. Customers who haven't bought in more than 6 months are considered churned.

In [ ]:
master_labelled = define_churn_label(master)

fig, ax = plt.subplots(figsize=(6, 4))
counts = master_labelled['churned'].value_counts().sort_index()
bars = ax.bar(['Active (0)', 'Churned (1)'], counts.values,
              color=['#5cb85c', '#d9534f'], width=0.5)
ax.set_title('Churn Label Distribution', fontsize=13)
ax.set_ylabel('Customer Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + val * 0.01,
            f'{val:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()